# 🔬 Experiment: SAM Segmentation $\rightarrow$ DAM Dense Captioning Step-by-Step Pipeline

This experiment tests and visualizes the end-to-end pipeline of generating high-precision **SAM (Meta Segment Anything Model ViT-B)** masks from organizer bounding boxes, and feeding those exact segmented masks into **DAM-3B (Describe Anything Model)** for localized text extraction (with a strict **50-word cap**).

**Key Highlights:**
- **Pure Meta SAM**: Uses official  directly on PyTorch with zero HuggingFace framework probing.
- **Step-by-Step Visualizations**: Shows (1) Frame + Bounding Boxes $\rightarrow$ (2) SAM Segmented Masks Overlay + IoU Scores $\rightarrow$ (3) Segmented Object Focus Crop + DAM-3B Generated Description.
- **Zero Cloud Syncing**: Runs locally on GPU memory without  / cloud sync.
- **50-word Cap**: Enforces 50 words max for clean semantic text embeddings.

In [ ]:
# 1. Safely step out to /kaggle/working, wipe old repo, and clone fresh
import os, sys
from pathlib import Path

if Path("/kaggle/working").exists():
    os.chdir("/kaggle/working")
    %cd /kaggle/working

!rm -rf /kaggle/working/AIC-2026
!git clone -b feature/dam-text-extraction https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git /kaggle/working/AIC-2026
%cd /kaggle/working/AIC-2026

if str(Path.cwd() / "src") not in sys.path:
    sys.path.insert(0, str(Path.cwd() / "src"))

print(f"✓ Current working directory: {Path.cwd()}")
!git log -1 --oneline


In [ ]:
# 2. Check GPU environment and install dependencies
import sys
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Install Meta official segment-anything & dependencies
!python -m pip install --quiet git+https://github.com/facebookresearch/segment-anything.git
!python -m pip install --quiet --no-deps -r requirements/kaggle.txt
!python -m pip install --quiet matplotlib


In [ ]:
# 3. Configure Dataset Paths and PathResolver
import os
from pathlib import Path

# Default Kaggle dataset roots (or fallback to local paths)
KEYFRAMES_ROOT = Path(os.environ.get("KEYFRAMES_ROOT", "/kaggle/input/datasets/lyduchoang/aic-26-video/Keyframes/Keyframes"))
OBJECTS_ROOT = Path(os.environ.get("OBJECTS_ROOT", "/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/objects"))
MAP_ROOT = Path(os.environ.get("MAP_ROOT", "/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/map-keyframes"))

from scripts.run_dam_batch import PathResolver

resolver = PathResolver(
    keyframes_root=KEYFRAMES_ROOT,
    objects_root=OBJECTS_ROOT,
    map_keyframes_root=MAP_ROOT,
)
print("✓ PathResolver initialized successfully!")

In [ ]:
# 4. Load Models: Meta SAM ViT-B (Segment Anything) and DAM-3B (Describe Anything)
import os, sys, urllib.request
from pathlib import Path
import torch

# Disable scipy/TF probing in transformers before importing DAM
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TORCH"] = "1"
try:
    import transformers.utils.import_utils as _t_import
    _t_import._scipy_available = False
    _t_import.is_scipy_available = lambda: False
    _t_import._sklearn_available = False
    _t_import.is_sklearn_available = lambda: False
    _t_import._tf_available = False
    _t_import.is_tf_available = lambda: False
    import transformers.utils as _t_utils
    _t_utils._scipy_available = False
    _t_utils.is_scipy_available = lambda: False
except Exception:
    pass

from segment_anything import sam_model_registry, SamPredictor
from aic2026.object_description.sam_backend import SamMaskGenerator
from aic2026.object_description.dam_backend import DamCaptioner

device = "cuda" if torch.cuda.is_available() else "cpu"
print("=" * 75)
print(f"🚀 [1/2] Loading Meta SAM (ViT-B) on {device}...")

# Download official Meta SAM checkpoint if not cached
checkpoint_path = Path("/kaggle/working/aic2026-model-cache/sam/sam_vit_b_01ec64.pth")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
if not checkpoint_path.exists() or checkpoint_path.stat().st_size < 100_000_000:
    print(f"📥 Downloading Meta SAM ViT-B checkpoint (375 MB) to {checkpoint_path}...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
        checkpoint_path,
    )
    print("✓ SAM checkpoint downloaded!")

sam = sam_model_registry["vit_b"](checkpoint=str(checkpoint_path)).to(device)
sam.eval()
sam_predictor = SamPredictor(sam)
sam_generator = SamMaskGenerator(backend_type="meta", predictor_or_processor=sam_predictor, device=device)
print("✓ Meta SAM loaded successfully!")

print(f"
🚀 [2/2] Loading DAM-3B (nvidia/DAM-3B) on {device}...")
dam_captioner = DamCaptioner.from_pretrained(
    model_id="nvidia/DAM-3B",
    revision="0797bedd98d645cd021379a4661ee233da279bba",
    code_revision="153ad3d33c29324e9197f565547c6bc8500da02d",
)
print("✓ DAM-3B loaded successfully!")
print("=" * 75)

In [ ]:
# 5. Define Visualization & Step-by-Step SAM -> DAM Pipeline Engine
from dataclasses import dataclass
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from aic2026.object_description import (
    load_organizer_detections,
    filter_detections,
    FilterConfig,
    normalized_to_pixels,
    normalize_caption,
)
from aic2026.object_description.rle import rectangle_mask

@dataclass
class MaskResult:
    mask: np.ndarray
    source: str
    iou_score: float | None

COLOR_PALETTE = [
    (0.96, 0.26, 0.21, 0.55),  # Red
    (0.13, 0.59, 0.95, 0.55),  # Blue
    (0.30, 0.69, 0.31, 0.55),  # Green
    (1.00, 0.76, 0.03, 0.55),  # Amber
    (0.61, 0.15, 0.69, 0.55),  # Purple
    (0.00, 0.74, 0.83, 0.55),  # Cyan
]

def plot_sam_masks_overlay(image: Image.Image, detections, sam_predictions, title: str = "SAM Segmented Masks"):
    """Render the full image with translucent SAM masks, contours, bounding boxes, and IoU scores."""
    w, h = image.size
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(image)
    
    overlay = np.zeros((h, w, 4), dtype=np.float32)
    for idx, (det, pred) in enumerate(zip(detections, sam_predictions)):
        color = COLOR_PALETTE[idx % len(COLOR_PALETTE)]
        mask = pred.mask
        overlay[mask] = color
        
        box = normalized_to_pixels(det.bbox_yxyx_norm, w, h)
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2.5,
            edgecolor=(color[0], color[1], color[2], 1.0),
            facecolor="none",
            linestyle="--",
        )
        ax.add_patch(rect)
        
        iou_str = f"IoU: {pred.iou_score:.3f}" if pred.iou_score is not None else "BBox Fallback"
        label_text = f"#{idx+1}: {det.class_entity} ({det.score:.2f}) | {iou_str}"
        ax.text(
            x1 + 4, max(15, y1 - 6),
            label_text,
            color="white",
            fontsize=10,
            weight="bold",
            bbox=dict(facecolor=(color[0], color[1], color[2], 0.85), edgecolor="none", pad=3, boxstyle="round,pad=0.3"),
        )
        
    ax.imshow(overlay)
    ax.set_title(title, fontsize=14, weight="bold", pad=12)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def plot_dam_region_cards(image: Image.Image, detections, sam_predictions, dam_captions, keyframe_n: int):
    """Display individual cards for each segmented region: Crop + Mask + DAM English Description."""
    w, h = image.size
    img_np = np.array(image.convert("RGB"))
    
    for idx, (det, pred, caption) in enumerate(zip(detections, sam_predictions, dam_captions)):
        color = COLOR_PALETTE[idx % len(COLOR_PALETTE)]
        box = normalized_to_pixels(det.bbox_yxyx_norm, w, h)
        x1, y1, x2, y2 = box
        
        masked_rgb = img_np.copy()
        masked_rgb[~pred.mask] = (20, 20, 20)
        
        pad_x = int((x2 - x1) * 0.15)
        pad_y = int((y2 - y1) * 0.15)
        crop_x1, crop_y1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
        crop_x2, crop_y2 = min(w, x2 + pad_x), min(h, y2 + pad_y)
        
        cropped_focus = img_np[crop_y1:crop_y2, crop_x1:crop_x2]
        cropped_mask = masked_rgb[crop_y1:crop_y2, crop_x1:crop_x2]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2))
        ax1.imshow(cropped_focus)
        ax1.set_title(f"Target Object #{idx+1}: {det.class_entity}", fontsize=11, weight="bold")
        ax1.axis("off")
        
        ax2.imshow(cropped_mask)
        iou_str = f"{pred.iou_score:.3f}" if pred.iou_score is not None else "BBox"
        ax2.set_title(f"SAM Mask Cutout (IoU: {iou_str})", fontsize=11, weight="bold")
        ax2.axis("off")
        
        plt.suptitle(f"Frame {keyframe_n} - Region #{idx+1} ({det.class_entity})", fontsize=13, weight="bold", y=1.02)
        plt.tight_layout()
        plt.show()
        
        print("┌" + "─" * 78 + "┐")
        print(f"│ 📝 DAM-3B ENGLISH CAPTION [{caption.word_count}/50 words | status: {caption.status}]".ljust(79) + "│")
        print("├" + "─" * 78 + "┤")
        import textwrap
        for line in textwrap.wrap(caption.description_en, width=74):
            print(f"│   {line.ljust(74)} │")
        print("└" + "─" * 78 + "┘
")


def run_sam_dam_experiment(
    video_id: str,
    max_frames_to_test: int = 3,
    score_threshold: float = 0.30,
    min_area_ratio: float = 0.005,
    max_area_ratio: float = 0.85,
    class_nms_iou: float = 0.45,
    max_regions_per_frame: int = 3,
    maximum_words: int = 50,
):
    """Full step-by-step interactive SAM -> DAM execution and visualization."""
    print("=" * 80)
    print(f" 🔬 RUNNING SAM -> DAM STEP-BY-STEP PIPELINE EXPERIMENT FOR: {video_id}")
    print(f"    • Score Threshold:    {score_threshold}")
    print(f"    • Max Regions/Frame:  {max_regions_per_frame}")
    print(f"    • Max Caption Words:  {maximum_words} words (strict cap)")
    print("=" * 80)
    
    filter_cfg = FilterConfig(
        minimum_score=score_threshold,
        minimum_area_ratio=min_area_ratio,
        maximum_area_ratio=max_area_ratio,
        same_class_iou=class_nms_iou,
        cross_label_duplicate_iou=0.60,
        maximum_regions=max_regions_per_frame,
    )
    
    frames_dir = resolver.resolve_frames_dir(video_id)
    objects_dir = resolver.resolve_objects_dir(video_id)
    
    json_files = sorted(objects_dir.glob("*.json"), key=lambda p: int(p.stem) if p.stem.isdigit() else 999999)
    if not json_files:
        print(f"❌ No object JSON files found in {objects_dir}")
        return
    
    tested_count = 0
    for json_path in json_files:
        keyframe_n = int(json_path.stem)
        
        img_candidates = [
            frames_dir / f"{keyframe_n:03d}.jpg",
            frames_dir / f"{keyframe_n:04d}.jpg",
            frames_dir / f"{keyframe_n}.jpg",
        ]
        img_path = next((p for p in img_candidates if p.exists()), None)
        if img_path is None:
            continue
            
        raw_dets = load_organizer_detections(json_path)
        filtered_dets = filter_detections(raw_dets, filter_cfg)
        if not filtered_dets:
            continue
            
        tested_count += 1
        print(f"
🎥 [Frame {tested_count}/{max_frames_to_test}] Keyframe #{keyframe_n} ({img_path.name})")
        print(f"    Raw Boxes: {len(raw_dets)} -> Filtered Distinct Objects: {len(filtered_dets)} ({[d.class_entity for d in filtered_dets]})")
        
        with Image.open(img_path) as src_img:
            image = src_img.convert("RGB")
            w, h = image.size
            rgb_np = np.array(image)
            
            boxes_xyxy = [normalized_to_pixels(d.bbox_yxyx_norm, w, h) for d in filtered_dets]
            
            # --- STEP 1: SAM SEGMENTATION ---
            print(f"    ⏳ [SAM] Starting Meta SAM segmentation for {len(boxes_xyxy)} bounding boxes...")
            if "sam_generator" in globals():
                sam_preds = sam_generator.generate(image, boxes_xyxy)
            else:
                sam_predictor.set_image(rgb_np)
                sam_preds = []
                for box in boxes_xyxy:
                    fallback_mask = rectangle_mask(h, w, box)
                    try:
                        masks, scores, _ = sam_predictor.predict(
                            box=np.array(box),
                            multimask_output=False,
                        )
                        mask = np.asarray(masks[0], dtype=bool)
                        score = float(scores[0])
                        sam_preds.append(MaskResult(mask=mask, source="sam", iou_score=score))
                    except Exception:
                        sam_preds.append(MaskResult(mask=fallback_mask, source="bbox_fallback", iou_score=None))
            
            iou_summary = [f"{p.iou_score:.3f}" if p.iou_score is not None else "fallback" for p in sam_preds]
            print(f"    ✅ [SAM] Finished SAM! Generated {len(sam_preds)} masks (IoU Scores: {iou_summary})")
            
            # Visualize SAM Masks Overlay on Full Image
            plot_sam_masks_overlay(
                image=image,
                detections=filtered_dets,
                sam_predictions=sam_preds,
                title=f"{video_id} - Keyframe #{keyframe_n} | Meta SAM Segmented Mask Overlays",
            )
            
            # --- STEP 2: DAM-3B DENSE CAPTIONING ---
            print(f"    ⏳ [DAM] Starting DAM-3B description for {len(sam_preds)} segmented regions (50-word cap)...")
            dam_captions = []
            for reg_idx, (det, pred) in enumerate(zip(filtered_dets, sam_preds), start=1):
                mask_pil = Image.fromarray(np.asarray(pred.mask, dtype=np.uint8) * 255, mode="L")
                raw_text = dam_captioner.describe(image, mask_pil, max_new_tokens=75)
                caption = normalize_caption(raw_text, maximum_words=maximum_words)
                dam_captions.append(caption)
                print(f"       • Region #{reg_idx} [{det.class_entity}]: "{caption.description_en}" ({caption.word_count} words)")
            print(f"    ✅ [DAM] Finished DAM descriptions for all {len(dam_captions)} regions!")
            
            # Display detailed SAM + DAM result cards
            plot_dam_region_cards(image, filtered_dets, sam_preds, dam_captions, keyframe_n)
            
        if tested_count >= max_frames_to_test:
            break
            
    print("=" * 80)
    print(f"🎉 EXPERIMENT COMPLETED FOR {video_id}! Processed {tested_count} sample keyframes.")
    print("=" * 80)

In [ ]:
# 6. Interactive Demo Launcher
# Choose any target video from the dataset to test SAM -> DAM pipeline interactively!
TARGET_VIDEO_ID = "L21_V001"

run_sam_dam_experiment(
    video_id=TARGET_VIDEO_ID,
    max_frames_to_test=3,
    score_threshold=0.30,
    max_regions_per_frame=3,
    maximum_words=50,  # Strict 50-word cap
)